## Import required libraries

In [ ]:
import pandas as pd
import tellurium as te
import sbmlnetwork
from Bio.KEGG.KGML import KGML_parser
from scipy.spatial.distance import euclidean
import requests
from bs4 import BeautifulSoup

## Load and Parse Model and Metadata

- Load an SBML model from an Antimony string using Tellurium.
- Load KEGG pathway XML for coordinates to map onto
- Load a CSV containing labels to use
- Load the CSV used to build the Antimony file, detailing all reactions

In [ ]:
# Load Antimony model and convert to SBML
r = te.loada(r'iMC057.txt')
net = sbmlnetwork.load(r.getSBML())

# Load KEGG XML and label mapping
xml_file = r'ko01100.xml'
pathway = KGML_parser.read(open(xml_file, 'r'))

In [ ]:
data = pd.read_csv(r'labels.csv', dtype='str', encoding='us-ascii', encoding_errors='ignore')
data = data.where(data.notnull(), None)

rxns = pd.read_csv(r'Reactions.csv')

## Map KEGG Reactions to Graphical Entry IDs

Map KEGG reaction IDs to their corresponding graphical entry IDs using orthology annotations from the XML file.

In [ ]:
def get_kegg_orthology_ids(reaction_ids):
    orthology_dict = {}
    for reaction_id in reaction_ids:
        url = f"https://www.kegg.jp/dbget-bin/www_bget?rn:{reaction_id}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the orthology section
        orthology_section = soup.find('th', string="Orthology")
        if orthology_section:
            # Get the orthology ID from the adjacent <td> tag
            orthology_ids = orthology_section.find_next('td').find_all('a')
            # Filter IDs that start with 'K'
            orthology_dict[reaction_id] = [ortho_id.get_text() for ortho_id in orthology_ids if ortho_id.get_text().startswith('K')]
        else:
            orthology_dict[reaction_id] = []
    
    return orthology_dict

In [ ]:
# orthology_ids = get_kegg_orthology_ids(rxns['Reaction ID'])

In [ ]:
orthology_ids = {'R00265': [],
 'R02767': ['K00059', 'K11539'],
 'R02768': ['K00647', 'K09458'],
 'R10707': ['K00648', 'K18473'],
 'R00238': ['K00626', 'K00632', 'K07508', 'K07509', 'K07513'],
 'R00742': ['K01946',
  'K01961',
  'K01962',
  'K01963',
  'K01964',
  'K02160',
  'K11262',
  'K11263',
  'K15036',
  'K15037',
  'K18472',
  'K18603',
  'K18604',
  'K18605',
  'K19312',
  'K22568'],
 'R01325': ['K01681', 'K01682', 'K27802'],
 'R01900': ['K01681', 'K01682', 'K27802'],
 'R00127': ['K00939', 'K18532', 'K18533'],
 'R00756': ['K00850', 'K16370', 'K21071', 'K24182'],
 'R10208': ['K01716', 'K02372', 'K16363', 'K22540', 'K22541'],
 'R00658': ['K01689', 'K27394'],
 'R01403': ['K00208', 'K00209', 'K02371'],
 'R01778': ['K00022', 'K07515'],
 'R02685': ['K07511', 'K07515'],
 'R07314': [],
 'R00762': ['K01086', 'K01622', 'K02446', 'K03841', 'K04041', 'K11532'],
 'R01068': ['K01622', 'K01623', 'K01624', 'K11645', 'K16305', 'K16306'],
 'R01082': ['K01675', 'K01676', 'K01677', 'K01678', 'K01679', 'K01774'],
 'R02740': ['K01810', 'K06859', 'K13810', 'K15916'],
 'R01061': ['K00134', 'K00150', 'K10705'],
 'R06185': [],
 'R00390': ['K01897', 'K15013', 'K28161'],
 'R00342': ['K00024', 'K00025', 'K00026'],
 'R01626': ['K00645', 'K00665', 'K00668', 'K11533', 'K13935'],
 'R01899': ['K00031'],
 'R00268': ['K00031'],
 'R00331': ['K00940', 'K18533'],
 'R00341': ['K01610'],
 'R00345': ['K01595'],
 'R00199': ['K01007'],
 'R08639': ['K01835', 'K15778'],
 'R01512': ['K00927'],
 'R01518': ['K01834', 'K01837', 'K15633', 'K15634', 'K15635'],
 'R00209': ['K00161', 'K00162', 'K00163', 'K00382', 'K00627'],
 'R00200': ['K00873', 'K12406'],
 'R01056': ['K01807', 'K01808'],
 'R01529': ['K01783'],
 'R02164': ['K00233',
  'K00234',
  'K00235',
  'K00236',
  'K00237',
  'K00239',
  'K00240',
  'K00241',
  'K00242',
  'K00244',
  'K00245',
  'K00246',
  'K00247',
  'K18859',
  'K18860',
  'K25801',
  'K25995',
  'K25996'],
 'R00405': ['K01902', 'K01903'],
 'R08575': ['K00616', 'K13810'],
 'R01641': ['K00615'],
 'R01015': ['K01803'],
 'R00351': ['K01647', 'K01659', 'K05942', 'K27797'],
 'R00479': ['K01637'],
 'R00235': ['K01895', 'K01913'],
 'R00945': ['K00600'],
 'R00315': ['K00925'],
 'R00132': ['K01672', 'K01673', 'K01674'],
 'R00220': ['K01752', 'K01754', 'K17989'],
 'R00703': ['K00016'],
 'R00835': ['K00036'],
 'R00344': ['K01958', 'K01959', 'K01960'],
 'R00519': ['K00122', 'K00123', 'K00124', 'K00126', 'K00127', 'K22515']}

In [ ]:
def map_reaction_to_entry_id(xml_file_path, reaction_orthology_dict):
    reaction_entry_map = {}
    
    for entry in pathway.entries.values():
        if entry.type != "ortholog":
            continue

        orthology_ids_in_entry = set(entry.name.replace('ko:', '').split())

        for reaction_id, orthologs in reaction_orthology_dict.items():
            if set(orthologs) & orthology_ids_in_entry:
                reaction_entry_map.setdefault(reaction_id, []).append(entry.id)
    
    return reaction_entry_map

reaction_dict = map_reaction_to_entry_id(xml_file, orthology_ids)

## Extract (x, y) Coordinates of Substrates and Products

- Extract graphical positions for compounds participating in reactions.
- Store positions keyed by compound ID, reaction ID, and entry ID.

In [ ]:
def extract_cpd_ids(rxn_str, label):
    """Extract cleaned compound IDs (without 'cpd:') from a reaction string under a given label."""
    try:
        line = next(l for l in rxn_str.splitlines() if label in l)
        part = line.split(f'{label}: ')[1]
        return [s.strip().replace('cpd:', '') for s in part.split(',')]
    except (StopIteration, IndexError):
        return []

def get_entry_coords(entry_id):
    """Safely return (x, y) coordinates of a pathway entry."""
    entry = pathway.entries.get(entry_id)
    if entry and entry.graphics:
        return entry.graphics[0].x, entry.graphics[0].y
    return None

# Load the pathway and initialize the map
cpd_coordinates_map = {}

for reaction_id, entry_ids in reaction_dict.items():
    found_any = False  # Track whether we processed any substrate/product

    for entry_id in entry_ids:
        reaction_entry = next((r for r in pathway.reactions if r.id == entry_id), None)
        if not reaction_entry:
            continue

        if reaction_id not in [rn.replace('rn:', '') for rn in reaction_entry._names]:
            continue

        rxn_str = str(reaction_entry)
        substrate_cpds = extract_cpd_ids(rxn_str, 'Substrates')
        product_cpds = extract_cpd_ids(rxn_str, 'Products')

        # ---- Substrates ----
        if reaction_entry._substrates:
            found_any = True
            for i, sid in enumerate(reaction_entry._substrates):
                if i < len(substrate_cpds):
                    coords = get_entry_coords(sid)
                    if coords:
                        if substrate_cpds[i] == 'C00080':
                            continue
                        key = f"{substrate_cpds[i]}_{reaction_id}_{entry_id}"
                        cpd_coordinates_map[key] = coords

        # ---- Products ----
        if reaction_entry._products:
            found_any = True
            for i, pid in enumerate(reaction_entry._products):
                if i < len(product_cpds):
                    coords = get_entry_coords(pid)
                    if coords:
                        if product_cpds[i] == 'C00080':
                            continue
                        key = f"{product_cpds[i]}_{reaction_id}_{entry_id}"
                        cpd_coordinates_map[key] = coords

    # After processing all entry_ids
    if not found_any:
        print(f"No substrate/product loops entered for {reaction_id}")

No substrate/product loops entered for R02768
No substrate/product loops entered for R01403
No substrate/product loops entered for R01778
No substrate/product loops entered for R02685
No substrate/product loops entered for R00132
No substrate/product loops entered for R02740
No substrate/product loops entered for R00390
No substrate/product loops entered for R00835


## Translate Compound IDs Using Custom Label Mapping

Replace KEGG compound IDs in the coordinate map with custom labels based on a CSV mapping.

In [ ]:
cpd_coordinates_map_tran = {}
for key, value in cpd_coordinates_map.items():
    new_key = key
    for _, row in data.iterrows():
        if row['KEGG ID'] in new_key:
            new_key = new_key.replace(row['KEGG ID'], row['ID'])
    cpd_coordinates_map_tran[new_key] = value

## Apply KEGG coordinates to SBMLNetwork layout

Step 1: Auto-layout the network to initialize positions

In [ ]:
net.auto_layout(max_num_connected_edges=1000)

Step 2: Load KEGG reaction mappings

In [ ]:
label_to_reaction_id = dict(zip(rxns['Label'], rxns['Reaction ID']))
reaction_id_to_label = {v: k for k, v in label_to_reaction_id.items()}

Step 3: Get current reaction IDs in network and map to KEGG IDs

In [ ]:
net_reactions = net.get_reactions_list()[0:57]
M3_reaction_labels = net.get_reaction_ids()[0:57]
M3_reactionIDs = [label_to_reaction_id.get(label, label) for label in M3_reaction_labels]

# Build label-to-reaction object dictionary
label_to_ID_dict = {
    reaction.get_reaction_id(): {'id': M3_reactionIDs[i], 'reaction_object': reaction}
    for i, reaction in enumerate(net_reactions)
}

Step 4: Translate KEGG-mapped coordinates

In [ ]:
translated_cpd_coordinates = {}
for key, coords in cpd_coordinates_map_tran.items():
    species, reaction, entry_id = key.rsplit("_", 2)
    try:
        reaction_label = reaction_id_to_label[reaction]
        size = net.get_species(species).get_size()
        updated_coords = (coords[0] - size[0]/2, coords[1] - size[1]/2)
        translated_cpd_coordinates[(species, reaction_label, entry_id)] = updated_coords
    except KeyError:
        continue

Step 5: Organize coordinate data for species/reactions/entries

In [ ]:
species_mapping = {}
reaction_mapping = {}
entry_mapping = {}

for (species, reaction, entry_id), coords in translated_cpd_coordinates.items():
    species_mapping.setdefault(species, {}).setdefault(reaction, {})[entry_id] = coords
    reaction_mapping.setdefault(reaction, {}).setdefault(species, {})[entry_id] = coords
    entry_mapping.setdefault(entry_id, {}).setdefault(reaction, []).append(species)

Step 6: Assign coordinates or handle aliases

In [ ]:
set_coordinates = {}
reaction_aliases = {}
multipleentryids = []
special_metabs = []

for spc in net.get_species_list():
    try:
        allcoords = species_mapping[spc.get_species_id()]
    except KeyError:
        spc.hide()
        continue

    # Determine if coordinates are all identical
    coordinates = {coord for subdict in allcoords.values() for coord in subdict.values()}
    all_same = len(coordinates) == 1

    if all_same:
        coord = next(iter(coordinates))
        if coord not in set_coordinates:
            spc.set_position(coord)
            set_coordinates[coord] = spc.get_species_id()
    else:
        special_metabs.append(spc.get_species_id())
        for reaction, entry_coords in allcoords.items():
            if len(entry_coords) > 1 or len(allcoords) > 1:
                multipleentryids.append(reaction)

# Identify duplicate reactions requiring aliases
duplicatereactions = list({
    rxn for rxn in multipleentryids
    for v in reaction_mapping[rxn].values()
    if len(v) > 1
})

## Setting module colors

In [ ]:
net.group_reactions(
    reactions=['R1', 'R7', 'R8', 'R20', 'R25', 'R27', 'R28', 'R41', 'R42', 'R46', 'R47'],
    color='#2A7F62'
)

net.group_reactions(
    reactions=['R10', 'R13', 'R18', 'R19', 'R21', 'R22', 'R23', 'R30', 'R32', 'R33', 'R34', 'R35', 'R37', 'R38', 'R45'],
    color='#8390FA'
)

net.group_reactions(
    reactions=['R39', 'R40', 'R43', 'R44', 'R54'],
    color='#FAC748'
)

net.group_reactions(
    reactions=['R2', 'R3', 'R4', 'R6', 'R11', 'R12', 'R14', 'R24', 'R26'],
    color='#F06C9B'
)

net.group_reactions(
    reactions=['R5', 'R15', 'R16', 'R17'],
    color='#B6244F'
)

net.group_reactions(
    reactions=['R31', 'R36', 'R48', 'R50', 'R53'],
    color='#FF9913'
)

net.group_reactions(
    reactions=['R49', 'R52'],
    color='#C8A2C8'
)

net.group_reactions(
    reactions=['R9', 'R29', 'R51'],
    color='#7F00FF'
)

net.group_reactions(
    reactions=['R55', 'R56', 'R57', 'R25'],
    color='black'
)

[Reaction(id=R55, index=0),
 Reaction(id=R56, index=0),
 Reaction(id=R57, index=0),
 Reaction(id=R25, index=0)]

## Setting reaction curves

In [ ]:
reactions = net.get_reactions_list()
reactions.set_thicknesses(30)

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]

In [ ]:
# Hide all reactions initially
net.get_reactions_list().hide()

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]

In [ ]:
# Build a nested dictionary: reaction_mapping[reaction][entry_id][species] = coords
reaction_mapping = {}
for (species, reaction, entry_id), coords in translated_cpd_coordinates.items():
    reaction_mapping.setdefault(reaction, {}).setdefault(entry_id, {})[species] = coords

In [ ]:
def get_species(entry_mapping, entry_id):
    """Return a unique list of species involved in a given entry ID."""
    species = set()
    if entry_id in entry_mapping:
        for reaction in entry_mapping[entry_id].values():
            species.update(reaction)
    return list(species)

def get_unique_coordinates(entry_mapping, species_mapping, entry_id):
    """Return ordered unique coordinates and species for a given entry ID."""
    species_list = get_species(entry_mapping, entry_id)
    coordinates = []
    seen = set()
    for species in species_list:
        if species in species_mapping:
            for reaction_data in species_mapping[species].values():
                if entry_id in reaction_data:
                    coord = reaction_data[entry_id]
                    if coord not in seen:
                        coordinates.append(coord)
                        seen.add(coord)
    return coordinates, species_list

In [ ]:
def find_curve_origin(coords_list, reference_point):
    """Return the index of the point closest to the reference point."""
    distances = [euclidean(reference_point, p) for p in coords_list]
    return distances.index(min(distances))

In [ ]:
species = net.get_species_list()

for s in species:
    s.set_shape('circle')
    s.set_size([30, 30])
    spcname = s.get_species_id()
    for _, row in data.iterrows():
        if spcname == row['ID']:
            s.set_text(row['Label'])

species.move_texts_by((30, 30))

[[True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 [True],
 

In [ ]:
id_to_label = pd.Series(data['ID'].values, index=data['KEGG ID']).to_dict()

for reaction, entries in reaction_mapping.items():
    for i, entry_id in enumerate(entries):

        for graphicsid in range(len(pathway.entries[int(entry_id)].graphics)):
            linecoords = pathway.entries[int(entry_id)].graphics[graphicsid].coords
            try:
                rxn = net.get_reactions_list(reaction)[i]
            except IndexError:
                # rxn = net.get_reaction(reaction).create_alias()
                continue

            # Set reaction node position
            mid_point = ((linecoords[0][0] + linecoords[-1][0]) / 2, 
                         (linecoords[0][1] + linecoords[-1][1]) / 2)
            rxn.set_position(mid_point)

            first_point, last_point = linecoords[0], linecoords[-1]
            involved_coords, involved_species = get_unique_coordinates(entry_mapping, species_mapping, entry_id)
            start_idx = find_curve_origin(involved_coords, first_point)
            end_idx = find_curve_origin(involved_coords, last_point)

            # Determine middle point for breaking
            middle_point = (linecoords[len(linecoords)//2] 
                            if len(linecoords) > 2 
                            else mid_point)

            for j, species_idx in enumerate([start_idx, end_idx]):
                species_id = involved_species[species_idx]
                curve = rxn.get_curves_list(rxn.get_species(species_id)[0])[0]
                curve.show()

                terminate = False
                current_index = 0

                # Define buffer zones around species coordinates
                coverage_regions = [
                    (coord[0] - 18, coord[0] + 18, coord[1] - 18, coord[1] + 18) 
                    for coord in involved_coords
                ]

                if j == 1:
                    linecoords.reverse()

                for k in range(len(linecoords) - 1):
                    if terminate:
                        break

                    start, end = linecoords[k], linecoords[k + 1]
                    if start == middle_point:
                        break

                    for x_min, x_max, y_min, y_max in coverage_regions:
                        if ((x_min <= end[0] <= x_max and y_min <= end[1] <= y_max) and
                            not (x_min < start[0] < x_max and y_min < start[1] < y_max)):
                            
                            if not any(
                                x_min <= linecoords[-1][0] <= x_max and
                                y_min <= linecoords[-1][1] <= y_max
                                for x_min, x_max, y_min, y_max in coverage_regions):
                                curve.add_segment(start, end, start, end)
                                break

                            adjusted_x = x_max if start[0] >= x_max else x_min
                            adjusted_y = y_min if start[1] <= y_min else y_max

                            if current_index == 0:
                                curve.remove_segment(0)

                            curve.add_segment(start, (adjusted_x, adjusted_y), start, (adjusted_x, adjusted_y))
                            terminate = True
                            break

                    if not terminate:
                        if current_index == 0:
                            segment = curve.get_segment(current_index)
                            segment.set_start(start)
                            segment.set_end(end)
                            segment.set_control_point_1(start)
                            segment.set_control_point_2(end)
                            curve.add_segment(start, end, start, end)
                        else:
                            curve.add_segment(start, end, start, end)

                    current_index += 1

In [ ]:
net.draw(r'Figures\MapDrawingColoredModules.pdf', update_network_extents=False)